In [14]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
import time
import sib_api_v3_sdk
import re
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime,timezone
import uuid 


In [15]:
load_dotenv(override=True)

True

In [16]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [17]:
# json extractor
def extract_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print("Still failed:", e)

    return None

In [18]:
def safe_generate(prompt):
    max_retries = 3

    for _ in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "Return only valid JSON"},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7
            )

            return response.choices[0].message.content

        except Exception as e:
            if "429" in str(e):
                print("⏳ Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
            else:
                print("Error:", e)
                return None

    print("Max retries reached")
    return None

In [28]:
def run_agent(lead):
    full_prompt = f"""
You are an elite SDR at Hamro Aadhiyan writing hyper-personalized cold emails to BSC.CSIT students in Nepal.

Your job is NOT to write marketing copy.
Your job is to write emails that feel like a real human noticed this specific student and wrote a short message.

---

COMPANY CONTEXT:
- BSC.CSIT notes, videos, past questions
- AI-based weak area detection for exam improvement
- Based in Biratnagar, Nepal

LEAD:
- Name: {lead['name']}
- Role: {lead['role']}
- Goal: {lead['goal']}

SENDER:
- Name: Prasanna Niroula
- Position: CEO, Hamro Aadhiyan
- Website: www.hamroaadhiyan.com

---

BANNED PHRASES (never use any of these):
- "I've been in your shoes"
- "I wanted to reach out"
- "I noticed you're a..."
- "we understand the importance"
- "we are committed to helping"
- "our comprehensive solutions"
- "let you know about"
- "hope this email finds you well"
- "that's why I'm reaching out"
- ANY variation of the above

---

GOOD EMAIL STRUCTURE:
1. Line 1 → Start with a specific pain point about {lead['goal']} — no intro, just jump in
   Example: "CSIT exams hit different when you don't know which chapters to prioritize."
2. Line 2 → One sentence on what changes with Hamro Aadhiyan — outcome focused, not feature focused
3. Line 3 → Soft curiosity-based CTA, not pushy
   Example: "Would it help if you knew exactly where your marks are slipping?"
4. Sign off → short and human

---

EMAIL TYPES:
1. professional → warm, direct, peer-to-peer. No corporate language.
2. humorous → relatable student struggle, light humor, conversational
3. concise → max 3 lines, punchy and direct

---

HTML FORMAT (use this EXACT structure for all 3 emails):

<div style="background:#f4f6f8;padding:32px 16px;font-family:Arial,sans-serif;">
  <div style="max-width:600px;margin:0 auto;background:#ffffff;border-radius:12px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
    
    <!-- Header bar -->
    <div style="background:#4F46E5;padding:20px 28px;">
      <span style="color:#ffffff;font-size:18px;font-weight:bold;">Hamro Aadhiyan</span>
      <span style="color:#c7d2fe;font-size:13px;margin-left:8px;">for BSC.CSIT students</span>
    </div>

    <!-- Body -->
    <div style="padding:28px;">
      <p style="font-size:20px;font-weight:bold;color:#1a1a1a;margin:0 0 16px;">Hi {lead['name']},</p>
      <p style="font-size:15px;line-height:1.7;color:#374151;margin:0 0 12px;">
        [FIRST LINE — specific pain point]
      </p>
      <p style="font-size:15px;line-height:1.7;color:#374151;margin:0 0 12px;">
        [SECOND LINE — outcome of using Hamro Aadhiyan]
      </p>
      <p style="font-size:15px;line-height:1.7;color:#374151;margin:0 0 24px;">
        [THIRD LINE — soft CTA]
      </p>

      <!-- CTA Button -->
      <a href="https://www.hamroaadhiyan.com" style="display:inline-block;background:#4F46E5;color:#ffffff;text-decoration:none;padding:12px 28px;border-radius:8px;font-size:15px;font-weight:bold;">
        Check Hamro Aadhiyan
      </a>
    </div>

    <!-- Signature -->
    <div style="padding:20px 28px;border-top:1px solid #f0f0f0;background:#fafafa;">
      <p style="margin:0;font-size:14px;font-weight:bold;color:#1a1a1a;">Prasanna Niroula</p>
      <p style="margin:4px 0 0;font-size:13px;color:#6b7280;">CEO, Hamro Aadhiyan</p>
      <a href="https://www.hamroaadhiyan.com" style="font-size:13px;color:#4F46E5;text-decoration:none;">www.hamroaadhiyan.com</a>
    </div>

  </div>
</div>

---

CRITICAL RULES:
- Fill in the 3 body lines naturally based on the lead's goal: {lead['goal']}
- Keep each body line to 1-2 sentences max
- The CTA button must always link to https://www.hamroaadhiyan.com
- Use the EXACT HTML structure above — only change the body text
- All CSS must be inline
- NO placeholders like "[FIRST LINE]" in output — replace with real content

---

OUTPUT RULES:
- NO markdown
- NO explanation  
- NO extra text
- ONLY valid JSON

FORMAT:
{{
  "emails": {{
    "professional": "<div>...</div>",
    "humorous": "<div>...</div>",
    "concise": "<div>...</div>"
  }}
}}
"""

    response = safe_generate(full_prompt)

    if response is None:
        print("Generation failed completely")
        return None

    print(response)

    data = extract_json(response)

    if data is None:
        print("JSON parsing failed")
        print("RAW OUTPUT:\n", response)
        return None

    return data

In [29]:
def evaluate_emails(emails):
    prompt = f"""
You are an expert SDR email formatter.

You must select the BEST email and return it EXACTLY as-is.

CRITICAL RULES:
- Output must be CLEAN HTML email only
- NO placeholders like "(button here)" or "..."  
- NO explanations or comments
- NO markdown
- NO extra text outside email
- Preserve structure exactly as given
- If email is not HTML, convert it into proper HTML

Emails:

Professional:
{emails['professional']}

Humorous:
{emails['humorous']}

Concise:
{emails['concise']}

Return ONLY JSON:

{{
  "final_email": "<clean full HTML email>"
}}
"""

    response = safe_generate(prompt)

    if response is None:
        print("Evaluation failed")
        return None

    print("RAW RESPONSE:", response)

    data = extract_json(response)

    if data is None:
        print("JSON parsing failed")
        return None

    return data

In [30]:
# send_email function for dynamically sending email
def send_email(to_email,subject, body):
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key['api-key']= os.getenv("BREVO_API_KEY")

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    ) 
    email = sib_api_v3_sdk.SendSmtpEmail(
        to=[{"email":to_email}],
        sender={
            "name":"Hamro Aadhiyan",
            "email":"anything@prasannaniroula.com.np"
        },
        subject= subject,
        html_content = body
    )
    try:
        response = api_instance.send_transac_email(email)
        print("Full response:", response)
        print("Message ID:", response.message_id)  # ← check this exact value
        return response
    except Exception as e:
        print("Error:", e)
        return None

In [31]:
# storing record of email forwarded
def store_record(data):
    with open("email_logs.ndjson","a") as f:
        f.write(json.dumps(data)+"\n")

In [32]:
def now():
    return datetime.now(timezone.utc).isoformat()


In [33]:
def create_email_record(lead, to_email, subject, body):
    return {
        "id": str(uuid.uuid4()),
        "name": lead["name"],
        "email": to_email,
        "subject": subject,
        "body": body,
        "status": {
            "sent": False,
            "delivered": False,
            "opened": False,
            "clicked": False,
            "replied": False,
            "bounced": False,
            "spam": False,
            "unsubscribed": False,
            "bounce_type": None,
            "message_id": None,
        },
        "timestamps": {
            "sent": None,
            "delivered": None,
            "opened": None,
            "clicked": None,
            "replied": None,
            "bounced": None,
            "spam": None,
        }
    }

In [34]:
# Tools
def run_sdr(lead, to_email):
    #generating emails
    result= run_agent(lead)
    if not result:
        print("Email generation failed")
        return
    
    emails = result["emails"]
    print(emails)

    #Emails selection
    result = evaluate_emails(emails)
    if not result:
        print("Evaluation failed")
        return
    
    final_email = result["final_email"]
    subject = f"Quick help for {lead['role']}"

    # Recording email 
    record = create_email_record(lead, to_email, subject, final_email)

    #sending email
    response = send_email(to_email = to_email , subject=subject, body= final_email)
    print(final_email)

    if response:
        record["status"]["sent"] = True
        record["timestamps"]["sent"]= now()
        record["status"]["message_id"] = response.message_id
        print("Email sent successfully")
    else:
        record["status"]["sent"]=False
        print("Email sending failed")
    
    #storing record
    store_record(record)






In [35]:
lead = {
    "name": "Prasanna",
    "role": "BSC.CSIT student",
    "goal": "improve exam performance",
}

run_sdr(
    lead=lead,
    to_email="prasannaniroula987@gmail.com"
)

{
  "emails": {
    "professional": "<div style=\"background:#f4f6f8;padding:32px 16px;font-family:Arial,sans-serif;\"><div style=\"max-width:600px;margin:0 auto;background:#ffffff;border-radius:12px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.08);\"><div style=\"background:#4F46E5;padding:20px 28px;\"><span style=\"color:#ffffff;font-size:18px;font-weight:bold;\">Hamro Aadhiyan</span><span style=\"color:#c7d2fe;font-size:13px;margin-left:8px;\">for BSC.CSIT students</span></div><div style=\"padding:28px;\"><p style=\"font-size:20px;font-weight:bold;color:#1a1a1a;margin:0 0 16px;\">Hi Prasanna,</p><p style=\"font-size:15px;line-height:1.7;color:#374151;margin:0 0 12px;\">CSIT exams can be tough when you're not sure which areas to focus on.</p><p style=\"font-size:15px;line-height:1.7;color:#374151;margin:0 0 12px;\">With Hamro Aadhiyan, you can identify your weak areas and improve your exam performance significantly.</p><p style=\"font-size:15px;line-height:1.7;color:#374151;marg